# ZerO: ResNet-18 / ResNet-50 и обучение

**Смысл статьи.** Случайность начальных весов можно заменить структурой: identity сохраняет сигнал, а преобразование Адамара при расширении каналов помогает избежать вырождения признаков. Последняя свёртка остаточной ветви получает нули; остальные — структурированные веса. Это не полное обнуление сети и не обнуление γ BatchNorm.

Для матрицы $W\in\mathbb{R}^{p\times q}$:
- при $p\le q$: частичная единичная матрица;
- при $p>q$: $W=\sqrt{2/N}\,H_N[:p,:q]$, $N=2^{\lceil\log_2 p\rceil}$.

У свёрток заполняется только центр ядра. $H_N$ содержит **+1 и −1**, поэтому веса включают отрицательные значения с масштабным коэффициентом, несмотря на название статьи. BatchNorm: γ=1, β=0.

Авторы также изучают траектории ранга и разброс качества между запусками. Детерминированные веса не устраняют случайность порядка батчей и аугментаций.

Источник: [Zhao et al., Algorithms 1–2, Sections 3–5](https://arxiv.org/pdf/2110.12661).

Здесь — проверяемое приближение к эксперименту CIFAR-10 с BatchNorm по версии arXiv v3. Полный рецепт ResNet авторы не опубликовали; неизвестные параметры отмечены ниже. Для обеих сетей используем stem 3×3 без начального pooling, чтобы сохранить разрешение 32×32. ImageNet и обучение без BN в этот эксперимент не входят. Pruning и Tucker-2 оцениваются после обучения отдельным `figure6.py`.

In [1]:
%pip install torch torchvision numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn

In [2]:
@torch.no_grad()
def hadamard_matrix(size, *, device=None, dtype=torch.float32):
    if size < 1 or size & (size - 1):
        raise ValueError("Размер Hadamard должен быть степенью двойки")
    h = torch.ones(1, 1, device=device, dtype=dtype)
    while h.shape[0] < size:
        h = torch.cat((torch.cat((h, h), dim=1),
                       torch.cat((h, -h), dim=1)), dim=0)
    return h


@torch.no_grad()
def structured_init_(weight, use_hadamard=True):
    if weight.ndim not in (2, 4):
        raise ValueError("Ожидается Linear [out, in] или Conv2d [out, in, kH, kW]")
    if weight.ndim == 4 and any(k % 2 == 0 for k in weight.shape[2:]):
        raise ValueError("Для центральной инициализации нужны нечётные размеры ядра")
    rows, cols = weight.shape[:2]
    if use_hadamard and rows > cols:
        size = 1 << (rows - 1).bit_length()
        matrix = hadamard_matrix(size, device=weight.device, dtype=weight.dtype)
        # Нормировка Algorithm 1 статьи: sqrt(2/N).
        matrix = matrix[:rows, :cols] * (2.0 / size) ** 0.5
    else:
        matrix = torch.eye(rows, cols, device=weight.device, dtype=weight.dtype)
    weight.zero_()
    if weight.ndim == 2:
        weight.copy_(matrix)
    else:
        weight[:, :, weight.shape[2] // 2, weight.shape[3] // 2].copy_(matrix)
    return weight


def zero_init_(weight):
    return structured_init_(weight, use_hadamard=True)


def identity_init_(weight):
    # Ablation: убираем Hadamard, сохраняя остальную схему ZerO.
    return structured_init_(weight, use_hadamard=False)

В [демонстрационном MNIST-ноутбуке авторов](https://github.com/jiaweizzhao/ZerO-initialization/blob/main/example_mnist.ipynb) используется $1/\sqrt{N}$; здесь явно выбрана формула $\sqrt{2/N}$ из Algorithm 1 PDF. Эти варианты отличаются множителем $\sqrt{2}$, их не следует незаметно смешивать.

Режимы: `init="zero"` — ZerO; `"identity"` — ablation без Hadamard; `"zeros"` — все Conv/Linear нулевые (отрицательный контроль); `"kaiming"`, `"xavier"`, `"normal"` или функция `init(weight)` — прежний интерфейс. `zero_init_residual=True` остаётся отдельной опцией для случайных инициализаций.

In [3]:
class ResBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        expanded_channels = out_channels * self.expansion
        self.conv1 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, expanded_channels, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(expanded_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != expanded_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, expanded_channels, 1, stride, bias=False),
                nn.BatchNorm2d(expanded_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return self.relu(out + identity)

In [4]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000, in_channels=3,
                 init="kaiming", zero_init_residual=False, cifar_stem=False):
        super().__init__()
        self._channels = 64
        self.conv1 = (nn.Conv2d(in_channels, 64, 3, 1, 1, bias=False) if cifar_stem
                      else nn.Conv2d(in_channels, 64, 7, 2, 3, bias=False))
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.Identity() if cifar_stem else nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        self.initialize_weights(init, zero_init_residual)

    def _make_layer(self, block, channels, count, stride=1):
        layers = [block(self._channels, channels, stride)]
        self._channels = channels * block.expansion
        layers.extend(block(self._channels, channels) for _ in range(1, count))
        return nn.Sequential(*layers)

    @torch.no_grad()
    def initialize_weights(self, init="kaiming", zero_init_residual=False):
        """Инициализировать Conv2d/Linear выбранным методом, BatchNorm — отдельно.

        init: 'zero', 'identity', 'kaiming', 'xavier', 'normal', 'zeros' или функция init(weight),
        изменяющая переданный тензор на месте. Bias обнуляются.
        Метод можно вызывать повторно для нового эксперимента.
        """
        if isinstance(init, str) and init in {"zero", "identity"} and zero_init_residual:
            raise ValueError("Для ZerO/identity оставьте zero_init_residual=False: обнуляется последняя Conv, а не BN")
        initializers = {
            "zero": zero_init_,
            "identity": identity_init_,
            "kaiming": lambda w: nn.init.kaiming_normal_(w, mode="fan_out", nonlinearity="relu"),
            "xavier": nn.init.xavier_normal_,
            "normal": lambda w: nn.init.normal_(w, mean=0.0, std=0.01),
            "zeros": nn.init.zeros_,
        }
        if isinstance(init, str):
            if init not in initializers:
                raise ValueError(f"Неизвестная инициализация {init!r}. Варианты: {list(initializers)}")
            initializer = initializers[init]
        elif callable(init):
            initializer = init
        else:
            raise TypeError("init должен быть строкой или функцией init(weight)")

        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                # Для линейного выхода в режиме Kaiming используем gain=1.
                if isinstance(init, str) and init == "kaiming" and isinstance(module, nn.Linear):
                    nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="linear")
                else:
                    initializer(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm2d):
                module.reset_running_stats()
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

        # Алгоритм ResNet из статьи: последняя свёртка нулевая, gamma BN = 1.
        if isinstance(init, str) and init in {"zero", "identity"}:
            for module in self.modules():
                if isinstance(module, ResBlock):
                    nn.init.zeros_(module.conv2.weight)
                elif isinstance(module, Bottleneck):
                    nn.init.zeros_(module.conv3.weight)

        if zero_init_residual:
            for module in self.modules():
                if isinstance(module, ResBlock):
                    nn.init.zeros_(module.bn2.weight)
                elif isinstance(module, Bottleneck):
                    nn.init.zeros_(module.bn3.weight)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = torch.flatten(self.avgpool(x), 1)
        return self.fc(x)


class ResNet18(ResNet):
    def __init__(self, num_classes=1000, in_channels=3, init="kaiming", zero_init_residual=False, cifar_stem=False):
        super().__init__(ResBlock, [2, 2, 2, 2], num_classes, in_channels, init, zero_init_residual, cifar_stem)


class ResNet50(ResNet):
    def __init__(self, num_classes=1000, in_channels=3, init="kaiming", zero_init_residual=False, cifar_stem=False):
        super().__init__(Bottleneck, [3, 4, 6, 3], num_classes, in_channels, init, zero_init_residual, cifar_stem)

## Проверка идеи до обучения

Первый тест проверяет независимость начальных весов от seed. Второй показывает, что после обновления нулевая последняя свёртка остаточной ветви может начать обучаться; все её предшественники не обязаны иметь ненулевой градиент на самом первом шаге.

In [5]:
def check_zero_initialization():
    for model_cls in (ResNet18, ResNet50):
        torch.manual_seed(1)
        first = model_cls(num_classes=10, init="zero", cifar_stem=True)
        torch.manual_seed(99)
        second = model_cls(num_classes=10, init="zero", cifar_stem=True)
        assert all(torch.equal(a, b) for a, b in zip(first.state_dict().values(), second.state_dict().values()))
        for block in first.modules():
            if isinstance(block, (ResBlock, Bottleneck)):
                last_conv = block.conv2 if isinstance(block, ResBlock) else block.conv3
                last_bn = block.bn2 if isinstance(block, ResBlock) else block.bn3
                assert torch.count_nonzero(last_conv.weight) == 0
                assert torch.all(last_bn.weight == 1)
        del second
        first.train()
        optimizer = torch.optim.SGD(first.parameters(), lr=0.01)
        images, labels = torch.randn(2, 3, 32, 32), torch.tensor([0, 1])
        target = first.layer4[-1]
        weight = target.conv2.weight if isinstance(target, ResBlock) else target.conv3.weight
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(first(images), labels)
        loss.backward()
        assert weight.grad is not None and torch.isfinite(weight.grad).all()
        assert weight.grad.abs().sum() > 0
        optimizer.step()
        assert weight.abs().sum() > 0
        print(f"{model_cls.__name__}: deterministic initialization / gradient / update OK")


check_zero_initialization()

ResNet18: deterministic initialization / gradient / update OK
ResNet50: deterministic initialization / gradient / update OK


## Конфигурация: H100 на два часа

По умолчанию `h100_2h`: ResNet-18, CIFAR-10, методы ZerO и Kaiming, один seed, целевые 200 эпох на метод. Budget — 110 минут от старта скрипта, остальное время оставлено на установку и выгрузку результатов. Это практический запуск для сравнения качества, а не оценка std статьи по 10 seed.

Начальный замер на вашей GPU оценивает стоимость полной эпохи с оценкой test. Если 200 эпох на оба метода не помещаются, `auto_fit_budget=True` выбирает одинаковый меньший бюджет (не ниже 50 эпох) до начала обучения. Реальное расписание сохраняется в `runtime_plan.json` / `run_config.json`. Для фиксированных 200 эпох: `auto_fit_budget=False`; ограничитель времени всё равно сохраняет возможность продолжения.

| Параметр | Значение | Основание |
|---|---|---|
| SGD: LR / momentum / decay | 0.1 / 0.9 / 1e-4 | Section 4 |
| Warmup | 10 эпох, старт 0.01 | Section 4 / Figure 8 |
| Целевой горизонт | 200 эпох | Figure 5, CIFAR-10 |
| Сравнение методов | одна архитектура, одинаковые seed/данные/бюджет | контролируемый эксперимент |
| Batch / форма warmup / decay | 128 / linear / cosine | детали ResNet не раскрыты авторами |
| Precision H100 | BF16, параметры и SGD-состояние FP32 | наше ускорение |
| CIFAR-10 | 50 000 train / 10 000 test | полный train, без дополнительного holdout |
| Checkpoint | последняя эпоха | не выбираем лучший по test |

Источник: [arXiv v3](https://arxiv.org/pdf/2110.12661v3). `paper_cifar` включает 3 метода × 10 seed, FP32, 200 эпох без ограничения времени; он не рассчитан на двухчасовую аренду. `cifar_both` дополнительно включает ResNet-50 на CIFAR-10 — отдельное сравнение, не ImageNet из Figure 5. `smoke` проверяет код на небольшом подмножестве.

Разбивка каналов, CIFAR-stem, конкретные варианты случайных initializers, аугментации, нормализация и развёртка ядра для stable rank — зафиксированные детали реализации; полного эталонного ResNet-кода в [репозитории авторов](https://github.com/jiaweizzhao/ZerO-initialization) нет. Поэтому точное совпадение чисел заранее не обещается.


In [ ]:
import csv
import json
import math
import random
import time
from contextlib import nullcontext
from dataclasses import dataclass, asdict, replace
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


@dataclass
class ExperimentConfig:
    architectures: tuple = ("resnet18",)
    methods: tuple = ("zero", "kaiming", "xavier")
    seeds: tuple = tuple(range(10))
    epochs: int = 200  # Горизонт CIFAR-10 на Figure 5, не отдельное указание в тексте.
    warmup_epochs: int = 10
    batch_size: int = 128  # Не указан авторами для ResNet.
    lr: float = 0.1
    warmup_start_lr: float = 0.01
    momentum: float = 0.9
    weight_decay: float = 1e-4
    val_size: int = 0  # В основном сравнении используем все 50 000 train.
    split_seed: int = 2026
    train_limit: int | None = None
    eval_limit: int | None = None
    num_workers: int = 8
    rank_every: int = 5
    data_dir: str = "data"
    output_dir: str = "runs/zero-init"
    device: str = "auto"
    precision: str = "fp32"  # H100-пресет использует bf16; paper_cifar — fp32.
    channels_last: bool = True
    budget_minutes: float | None = None
    auto_fit_budget: bool = False
    min_epochs: int = 50
    benchmark_batches: int = 30
    log_every: int = 100
    cpu_threads: int = 4
    # Неизвестные детали статьи явно сохраняем в config.json.
    dataset: str = "cifar10"
    lr_schedule: str = "linear_warmup_cosine"  # Decay в статье не раскрыт.
    checkpoint_selection: str = "last_epoch"
    paper_version: str = "2110.12661v3"


paper_config = ExperimentConfig()
h100_config = replace(paper_config, methods=("zero", "kaiming"), seeds=(0,),
                      precision="bf16", budget_minutes=110, auto_fit_budget=True)
smoke_config = replace(paper_config, architectures=("resnet18",), methods=("zero", "kaiming"),
                       seeds=(0,), epochs=2, warmup_epochs=1, batch_size=32,
                       train_limit=512, eval_limit=256, num_workers=0, rank_every=1)
PRESETS = {"h100_2h": h100_config, "paper_cifar": paper_config, "full": paper_config,
           "cifar_both": replace(paper_config, architectures=("resnet18", "resnet50")),
           "smoke": smoke_config}
PRESET = "h100_2h"
config = PRESETS[PRESET]


def select_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # На одной платформе фиксируем порядок данных; GPU не обещает битовую повторяемость.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def amp_context(cfg, device):
    if cfg.precision == "bf16":
        if device.type != "cuda" or not torch.cuda.is_bf16_supported():
            raise ValueError("bf16-пресет требует CUDA с BF16 (например H100); задайте precision='fp32'")
        return torch.autocast("cuda", dtype=torch.bfloat16)
    if cfg.precision != "fp32":
        raise ValueError("precision: fp32 или bf16")
    return nullcontext()


def configure_runtime(cfg):
    torch.set_num_threads(cfg.cpu_threads)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    device = select_device(cfg.device)
    # Проверка до загрузки данных / дорогого создания сети.
    with amp_context(cfg, device):
        pass
    return device


print(PRESET, asdict(config))
print("Device:", select_device(config.device))


## CIFAR-10

Используем все 50 000 train и 10 000 test. Crop с padding=4 и horizontal flip применяются только к train. Архив загружается с зеркала MIT Open Learning; перед распаковкой проверяется оригинальный MD5, сами файлы также проверяются torchvision. Test служит мониторингом фиксированного запуска: по нему не выбираются checkpoint или гиперпараметры.


In [ ]:
def ensure_cifar10(root):
    # Зеркало содержит тот же архив: torchvision проверяет исходный MD5.
    # Верификация обязательна также перед распаковкой и чтением pickle.
    from torchvision.datasets.utils import check_integrity, download_and_extract_archive
    root = Path(root)
    expected = datasets.CIFAR10.train_list + datasets.CIFAR10.test_list
    files_ok = all(check_integrity(str(root / "cifar-10-batches-py" / name), md5)
                   for name, md5 in expected)
    meta = datasets.CIFAR10.meta
    files_ok = files_ok and check_integrity(str(root / "cifar-10-batches-py" / meta["filename"]), meta["md5"])
    if not files_ok:
        download_and_extract_archive(
            "https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/resolve/main/cifar-10-python.tar.gz",
            str(root), filename="cifar-10-python.tar.gz", md5=datasets.CIFAR10.tgz_md5,
        )


def make_loaders(cfg, seed):
    if cfg.dataset != "cifar10":
        raise ValueError("Этот протокол поддерживает CIFAR-10; ImageNet — отдельный эксперимент")
    ensure_cifar10(cfg.data_dir)
    normalize = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), normalize,
    ])
    eval_transform = transforms.Compose([transforms.ToTensor(), normalize])
    train_data = datasets.CIFAR10(cfg.data_dir, train=True, download=False, transform=train_transform)
    test_data = datasets.CIFAR10(cfg.data_dir, train=False, download=False, transform=eval_transform)
    if not 0 <= cfg.val_size < len(train_data):
        raise ValueError("val_size должен быть между 0 и размером train - 1")
    generator = torch.Generator().manual_seed(cfg.split_seed)
    indices = torch.randperm(len(train_data), generator=generator).tolist()
    val_indices, train_indices = indices[:cfg.val_size], indices[cfg.val_size:]
    if cfg.train_limit is not None:
        train_indices = train_indices[:cfg.train_limit]
    test_indices = list(range(len(test_data)))
    if cfg.eval_limit is not None:
        val_indices = val_indices[:cfg.eval_limit]
        test_indices = test_indices[:cfg.eval_limit]
    common = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers,
                  pin_memory=select_device(cfg.device).type == "cuda",
                  persistent_workers=False)  # Явные seeds на каждую эпоху, включая resume.
    if cfg.num_workers > 0:
        common["prefetch_factor"] = 4
    train_loader = DataLoader(Subset(train_data, train_indices), shuffle=True,
                              generator=torch.Generator().manual_seed(seed), **common)
    val_loader = None
    if val_indices:
        val_data = datasets.CIFAR10(cfg.data_dir, train=True, download=False, transform=eval_transform)
        val_loader = DataLoader(Subset(val_data, val_indices), shuffle=False,
                                generator=torch.Generator().manual_seed(seed + 1), **common)
    test_loader = DataLoader(Subset(test_data, test_indices), shuffle=False,
                             generator=torch.Generator().manual_seed(seed + 2), **common)
    return train_loader, val_loader, test_loader


def create_model(architecture, method, cfg=None):
    classes = {"resnet18": ResNet18, "resnet50": ResNet50}
    model = classes[architecture](num_classes=10, init=method, cifar_stem=True)
    if cfg is not None and cfg.channels_last:
        model = model.to(memory_format=torch.channels_last)
    return model


## Метрики и цикл обучения

Stable rank: $\|W\|_F^2/\sigma_{max}^2(W)$, ядро разворачивается в $[C_{out}, C_{in}k_Hk_W]$. Записываем `layer2[0].conv1`, `layer3[0].conv1`, `layer4[0].conv1` раз в `rank_every` эпох. Для нулевой матрицы метрика равна 0. Эта развёртка — явное соглашение реализации; форму опубликованных кривых не подгоняем.

На H100 применяются BF16 autocast и channels-last; CPU-синхронизация не выполняется на каждом батче. Train/test loss и accuracy считаются с весом размера батча. `last.pt` атомарно сохраняет веса, momentum SGD и номер полной эпохи; состояние генераторов данных определяется парой seed/epoch, включая resume. При ограничении времени незавершённая эпоха пересчитывается из последнего checkpoint.

Метрики `summary.json` относятся только к завершённому расписанию и последней эпохе. У прерванного запуска остаются `history.json`, `last.pt` и статус `time_limit`; он не маскируется под полный результат.


In [ ]:
@torch.no_grad()
def stable_ranks(model):
    result = {}
    for name in ("layer2", "layer3", "layer4"):
        w = getattr(model, name)[0].conv1.weight.detach().float().cpu().flatten(1)
        s = torch.linalg.svdvals(w)
        spectral_sq = s[0].square().item()
        result[name] = float(w.square().sum().item() / spectral_sq) if spectral_sq else 0.0
    return result


def learning_rate_at(step, total_steps, warmup_steps, peak_lr, warmup_start_lr=0.01):
    if step < warmup_steps:
        fraction = step / max(1, warmup_steps - 1)
        return warmup_start_lr + (peak_lr - warmup_start_lr) * fraction
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps - 1)
    return peak_lr * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def run_epoch(model, loader, device, optimizer=None, cfg=None, epoch=0,
              deadline=None, max_batches=None):
    training = optimizer is not None
    model.train(training)
    # Без .item() на каждом батче: меньше синхронизаций CPU/CUDA.
    totals = torch.zeros(3, device=device, dtype=torch.float32)
    batches = 0
    started = time.perf_counter()
    with torch.set_grad_enabled(training):
        for batch_index, (images, labels) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            if deadline is not None and time.monotonic() >= deadline:
                raise TimeoutError("Лимит времени: продолжите с last.pt последней полной эпохи")
            memory_format = torch.channels_last if cfg.channels_last else torch.contiguous_format
            images = images.to(device, non_blocking=True, memory_format=memory_format)
            labels = labels.to(device, non_blocking=True)
            if training:
                step = epoch * len(loader) + batch_index
                lr = learning_rate_at(step, cfg.epochs * len(loader),
                                      cfg.warmup_epochs * len(loader), cfg.lr, cfg.warmup_start_lr)
                for group in optimizer.param_groups:
                    group["lr"] = lr
                optimizer.zero_grad(set_to_none=True)
            with amp_context(cfg, device):
                logits = model(images)
                loss = nn.functional.cross_entropy(logits, labels)
            if training:
                # BF16 обладает диапазоном FP32; GradScaler для BF16 не нужен.
                loss.backward()
                optimizer.step()
            batch_size = labels.size(0)
            totals[0] += loss.detach().float() * batch_size
            totals[1] += (logits.argmax(1) == labels).sum()
            totals[2] += batch_size
            batches += 1
            if training and max_batches is None and cfg.log_every and batches % cfg.log_every == 0:
                loss_sum, correct, count = totals.tolist()
                if not math.isfinite(loss_sum):
                    raise FloatingPointError(f"Loss не конечен: epoch={epoch + 1}, batch={batches}")
                print(f"  epoch={epoch + 1} batch={batches}/{len(loader)} "
                      f"loss={loss_sum/count:.4f} acc={100*correct/count:.2f}%", flush=True)
    loss_sum, correct, count = totals.tolist()
    if count == 0:
        raise ValueError("Пустой DataLoader")
    if not math.isfinite(loss_sum):
        raise FloatingPointError(f"Loss не конечен: epoch={epoch + 1}")
    synchronize(device)
    return {"loss": loss_sum / count, "accuracy": 100.0 * correct / count,
            "images": int(count), "batches": batches, "seconds": time.perf_counter() - started}


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)


def save_history(history, directory):
    write_json(directory / "history.json", history)
    with (directory / "history.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)


def cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {k: cpu_tree(v) for k, v in value.items()}
    if isinstance(value, list):
        return [cpu_tree(v) for v in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(v) for v in value)
    return value


def save_checkpoint(model, optimizer, epoch, cfg, directory):
    path = directory / "last.pt"
    temporary = directory / "last.pt.tmp"
    torch.save({"model_state": cpu_tree(model.state_dict()),
                "optimizer_state": cpu_tree(optimizer.state_dict()),
                "epoch": epoch, "config": asdict(cfg)}, temporary)
    temporary.replace(path)


def make_optimizer(model, cfg):
    return torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum,
                           weight_decay=cfg.weight_decay, foreach=True)


def benchmark_runtime(cfg, root, deadline):
    """Замер скорости; эти обновления НЕ входят в результаты эксперимента."""
    device = select_device(cfg.device)
    profiles = {}
    train_loader, _, test_loader = make_loaders(cfg, cfg.seeds[0])
    for architecture in cfg.architectures:
        seed_everything(cfg.seeds[0])
        model = create_model(architecture, cfg.methods[0], cfg).to(device)
        optimizer = make_optimizer(model, cfg)
        run_epoch(model, train_loader, device, optimizer, cfg, deadline=deadline, max_batches=5)
        train = run_epoch(model, train_loader, device, optimizer, cfg,
                          deadline=deadline, max_batches=cfg.benchmark_batches)
        test = run_epoch(model, test_loader, device, cfg=cfg, deadline=deadline, max_batches=10)
        started = time.perf_counter()
        stable_ranks(model)
        rank_seconds = time.perf_counter() - started
        raw_epoch = (train["seconds"] / train["batches"] * len(train_loader)
                     + test["seconds"] / test["batches"] * len(test_loader)
                     + rank_seconds / cfg.rank_every + 1.0)
        # Запас на checkpoint, IO и колебания пропускной способности.
        profiles[architecture] = {"estimated_epoch_seconds": raw_epoch * 1.25,
                                  "train_images_per_second": train["images"] / train["seconds"]}
        print("Benchmark", architecture, profiles[architecture], flush=True)
        del optimizer, model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    seconds_per_shared_epoch = sum(p["estimated_epoch_seconds"] for p in profiles.values()) * len(cfg.methods) * len(cfg.seeds)
    remaining = deadline - time.monotonic() - 180  # Финальные графики / IO.
    epochs_that_fit = max(0, int(remaining / seconds_per_shared_epoch))
    chosen = min(cfg.epochs, epochs_that_fit)
    if chosen < cfg.epochs:
        chosen = chosen // 10 * 10
    plan = {"profiles": profiles, "requested_epochs": cfg.epochs,
            "chosen_epochs": chosen if cfg.auto_fit_budget else cfg.epochs,
            "remaining_seconds": max(0, remaining), "auto_fit_budget": cfg.auto_fit_budget}
    write_json(root / "runtime_plan.json", plan)
    if cfg.auto_fit_budget:
        if chosen < max(cfg.min_epochs, cfg.warmup_epochs + 1):
            raise RuntimeError(f"За оставшееся время помещается лишь {chosen} эпох на метод. "
                               "Увеличьте бюджет / скорость GPU; обучение не запущено.")
        print(f"Общий бюджет для каждого метода: {chosen} эпох (запрошено {cfg.epochs})", flush=True)
        cfg = replace(cfg, epochs=chosen)
    return cfg


def train_one(cfg, architecture, method, seed, directory, loaders=None, deadline=None):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    if cfg.epochs < 1 or not 0 <= cfg.warmup_epochs <= cfg.epochs or cfg.rank_every < 1:
        raise ValueError("Нужны epochs>=1, 0<=warmup_epochs<=epochs, rank_every>=1")
    if not 0 <= cfg.warmup_start_lr <= cfg.lr or cfg.lr <= 0:
        raise ValueError("Нужны lr>0 и 0<=warmup_start_lr<=lr")
    seed_everything(seed)
    train_loader, val_loader, test_loader = loaders if loaders is not None else make_loaders(cfg, seed)
    device = select_device(cfg.device)
    model = create_model(architecture, method, cfg).to(device)
    optimizer = make_optimizer(model, cfg)
    metadata = {**asdict(cfg), "architecture": architecture, "method": method, "seed": seed,
                "actual_device": str(device), "torch_version": str(torch.__version__),
                "torchvision_version": str(torchvision.__version__),
                "sizes": {"train": len(train_loader.dataset),
                          "val": len(val_loader.dataset) if val_loader is not None else 0,
                          "test": len(test_loader.dataset)}}
    if (directory / "config.json").exists():
        previous = json.loads((directory / "config.json").read_text())
        keys = ("architecture", "method", "seed", "epochs", "warmup_epochs", "batch_size", "lr",
                "warmup_start_lr", "momentum", "weight_decay", "precision", "val_size", "split_seed",
                "train_limit", "eval_limit", "num_workers", "dataset", "lr_schedule")
        changed = [key for key in keys if previous.get(key) != metadata.get(key)]
        if changed:
            raise ValueError(f"Нельзя менять протокол существующего запуска: {changed}")
    metadata["device_name"] = torch.cuda.get_device_name(device) if device.type == "cuda" else str(device)
    write_json(directory / "config.json", metadata)
    first_epoch = 0
    if (directory / "last.pt").exists():
        checkpoint = torch.load(directory / "last.pt", map_location="cpu", weights_only=True)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        first_epoch = checkpoint["epoch"]
        history = [r for r in json.loads((directory / "history.json").read_text()) if r["epoch"] <= first_epoch]
        del checkpoint
        print(f"Resume {architecture}/{method}: epoch {first_epoch}", flush=True)
    else:
        initial_test = run_epoch(model, test_loader, device, cfg=cfg, deadline=deadline)
        history = [{"epoch": 0, "lr": 0.0, "train_loss": None, "train_accuracy": None,
                    "val_loss": None, "val_accuracy": None,
                    "test_loss": initial_test["loss"], "test_accuracy": initial_test["accuracy"],
                    "seconds": 0.0, **{f"rank_{k}": v for k, v in stable_ranks(model).items()}}]
        save_history(history, directory)
        save_checkpoint(model, optimizer, 0, cfg, directory)
    for epoch in range(first_epoch, cfg.epochs):
        # Epoch-specific RNG делает возобновление независимым от предыдущего процесса.
        epoch_seed = seed * 100000 + epoch
        seed_everything(epoch_seed)
        if train_loader.generator is not None:
            train_loader.generator.manual_seed(epoch_seed)
        started = time.perf_counter()
        write_json(directory / "status.json", {"status": "running", "epoch": epoch, "target_epochs": cfg.epochs})
        try:
            train = run_epoch(model, train_loader, device, optimizer, cfg, epoch, deadline)
            test = run_epoch(model, test_loader, device, cfg=cfg, deadline=deadline)
            val = run_epoch(model, val_loader, device, cfg=cfg, deadline=deadline) if val_loader is not None else None
        except TimeoutError:
            write_json(directory / "status.json", {"status": "time_limit", "completed_epochs": epoch,
                                                   "target_epochs": cfg.epochs})
            raise
        ranks = stable_ranks(model) if (epoch + 1) % cfg.rank_every == 0 or epoch + 1 == cfg.epochs else dict.fromkeys(("layer2", "layer3", "layer4"))
        row = {"epoch": epoch + 1, "lr": optimizer.param_groups[0]["lr"],
               "train_loss": train["loss"], "train_accuracy": train["accuracy"],
               "val_loss": val["loss"] if val else None, "val_accuracy": val["accuracy"] if val else None,
               "test_loss": test["loss"], "test_accuracy": test["accuracy"],
               "seconds": time.perf_counter() - started, **{f"rank_{k}": v for k, v in ranks.items()}}
        history.append(row)
        save_history(history, directory)
        save_checkpoint(model, optimizer, epoch + 1, cfg, directory)
        print(f"{architecture}/{method}/seed={seed} epoch={epoch + 1}/{cfg.epochs} "
              f"loss={train['loss']:.4f} test_acc={test['accuracy']:.2f}% "
              f"lr={row['lr']:.5f} time={row['seconds']:.1f}s", flush=True)
    final = history[-1]
    summary = {"architecture": architecture, "method": method, "seed": seed, "epochs": cfg.epochs,
               "test_loss": final["test_loss"], "test_accuracy": final["test_accuracy"],
               "test_error": 100.0 - final["test_accuracy"], "checkpoint_selection": "last_epoch",
               "precision": cfg.precision, "train_images": len(train_loader.dataset),
               "test_images": len(test_loader.dataset), "directory": str(directory), "status": "complete"}
    write_json(directory / "summary.json", summary)
    write_json(directory / "status.json", {"status": "complete", "completed_epochs": cfg.epochs})
    return summary


def run_experiments(cfg, resume=None):
    started = time.monotonic()
    budget_minutes = cfg.budget_minutes
    deadline = started + budget_minutes * 60 if budget_minutes is not None else None
    root = Path(resume) if resume is not None else Path(cfg.output_dir) / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    if resume is not None:
        # Расписание/число эпох при resume не меняем; бюджет относится к новой сессии.
        saved = json.loads((root / "run_config.json").read_text())
        for field in ("architectures", "methods", "seeds"):
            saved[field] = tuple(saved[field])
        cfg = replace(ExperimentConfig(**saved), device=cfg.device,
                      data_dir=cfg.data_dir, budget_minutes=budget_minutes)
    else:
        root.mkdir(parents=True, exist_ok=False)
        write_json(root / "requested_config.json", asdict(cfg))
    device = configure_runtime(cfg)
    print("Results:", root.resolve(), "device:", device, flush=True)
    print("Configuration:", asdict(cfg), flush=True)
    if resume is None:
        if deadline is not None:
            cfg = benchmark_runtime(cfg, root, deadline)
        write_json(root / "run_config.json", asdict(cfg))
    # 60 секунд остаётся на сохранение статуса / построение графиков.
    training_deadline = deadline - 60 if deadline is not None else None
    summaries = []
    try:
        for architecture in cfg.architectures:
            for seed in cfg.seeds:
                for method in cfg.methods:
                    directory = root / f"{architecture}-{method}-seed{seed}"
                    if (directory / "summary.json").exists():
                        summary = json.loads((directory / "summary.json").read_text())
                    else:
                        summary = train_one(cfg, architecture, method, seed, directory, deadline=training_deadline)
                    summaries.append(summary)
                    with (root / "summary.csv").open("w", newline="") as f:
                        writer = csv.DictWriter(f, fieldnames=list(summaries[0]))
                        writer.writeheader()
                        writer.writerows(summaries)
    except TimeoutError as error:
        print(str(error), flush=True)
        write_json(root / "status.json", {"status": "time_limit", "completed_runs": len(summaries),
                                          "resume_directory": str(root)})
        return root, summaries
    write_json(root / "status.json", {"status": "complete", "completed_runs": len(summaries)})
    return root, summaries


## Запуск на арендованной GPU

Обычный Run All использует выбранный пресет. Для долгого запуска из терминала рядом лежит `run_zero.py`: он исполняет только помеченные ячейки определений этого ноутбука, поэтому модель и обучение имеют один источник кода.

```bash
python run_zero.py --preset h100_2h --device cuda
```

Продолжение в новой сессии (архитектура и расписание читаются из сохранённого config):

```bash
python run_zero.py --resume runs/zero-init/ИМЯ_КАТАЛОГА --device cuda --budget-minutes 110
```

Для локальной проверки без CUDA: `python run_zero.py --preset smoke --device cpu`.


In [10]:
results_dir, summaries = run_experiments(config)

Results: /Users/ilya/Documents/compet/itmo/efficient-nn-hw/hw2/runs/zero-init/20261009-191600-043071


URLError: <urlopen error [Errno 8] nodename nor servname provided, or not known>

## Графики в стиле статьи

Основная фигура повторяет компоновку [Figure 5 (стр. 10)](https://arxiv.org/pdf/2110.12661#page=10): 2×3 панели stable rank. Верхний ряд — ResNet-18, нижний — ResNet-50; столбцы — первые свёртки групп 2, 3, 4. ZerO — синий, Kaiming — зелёный, Xavier — красный; светлый серо-сиреневый фон и белая сетка. Loss/accuracy вынесены в отдельные фигуры.

Кривые строятся из наших `history.json`, без сглаживания или подгонки к статье. В нашем эксперименте обе сети используют CIFAR-10; у авторов нижний ряд Figure 5 относится к ImageNet. Диапазоны осей определяются фактическими значениями. При `h100_2h` и `paper_cifar` нижний ряд останется пустым: ResNet-50 не входит в эти пресеты. Для отдельного сравнения обеих сетей есть `cifar_both`.

По умолчанию показаны средние линии без полос, как в статье; `show_std=True` добавляет ±1 выборочное стандартное отклонение. Среднее считается на общих записанных эпохах всех seed данного метода. Для более подробных траекторий перед обучением задайте `config = replace(config, rank_every=1)` (SVD станет чаще).

Сохраняются `figure5-stable-rank.png/.pdf` и отдельные `<architecture>-training-curves.png/.pdf`; PNG — 300 dpi, PDF — векторный. Графики доступны и для незавершённого запуска. Figure 3 и 4 требуют отдельных экспериментов (MLP, глубина сети). Для Figure 6 используйте `figure6.py` после обучения, как описано ниже.

`aggregate.csv` отдельно содержит измеренный test error, опубликованный ориентир Table 2 и разницу в процентных пунктах. Наличие ориентира не означает воспроизведения; один seed не даёт оценки разброса.


In [ ]:
# Оформление Figure 5; значения берутся только из history.json.
PAPER_STYLE = {
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.facecolor": "#EAEAF2", "figure.facecolor": "white",
    "axes.edgecolor": "white", "axes.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "white", "grid.linewidth": 0.7,
    "xtick.color": "#262626", "ytick.color": "#262626",
    "xtick.major.size": 0, "ytick.major.size": 0,
    "legend.frameon": False, "legend.fontsize": 8,
    "lines.linewidth": 1.5, "pdf.fonttype": 42,
}
METHOD_STYLES = {
    "zero": ("ZerO Init", "#0072BD", "-"),
    "kaiming": ("Kaiming Init (Random)", "#00A000", "-"),
    "xavier": ("Xavier Init (Random)", "#FF0000", "-"),
    "identity": ("Identity Init", "#9467BD", "--"),
    "zeros": ("All-zero Init", "#555555", ":"),
    "normal": ("Normal Init (Random)", "#E69F00", "--"),
}


def load_plot_runs(root):
    # Работает и во время обучения, до появления summary.json.
    runs = []
    for history_path in sorted(Path(root).glob("*/history.json")):
        directory = history_path.parent
        cfg_path = directory / "config.json"
        if not cfg_path.exists():
            continue
        cfg = json.loads(cfg_path.read_text())
        summary_path = directory / "summary.json"
        runs.append({"config": cfg, "history": json.loads(history_path.read_text()),
                     "summary": json.loads(summary_path.read_text()) if summary_path.exists() else None})
    if not runs:
        raise ValueError(f"В {root} нет history.json с config.json. Сначала запустите обучение.")
    return runs


def plot_metric(ax, runs, metric, show_std=False):
    available = {r["config"]["method"] for r in runs}
    methods = [m for m in METHOD_STYLES if m in available] + sorted(available - METHOD_STYLES.keys())
    for method in methods:
        selected = [r for r in runs if r["config"]["method"] == method]
        by_epoch = []
        for run in selected:
            by_epoch.append({row["epoch"]: float(row[metric]) for row in run["history"]
                             if row.get(metric) is not None and np.isfinite(row[metric])})
        # Одинаковый набор seed в каждой точке; без интерполяции пропусков.
        epochs = sorted(set.intersection(*(set(values) for values in by_epoch)))
        if not epochs:
            continue
        samples = np.array([[values[e] for e in epochs] for values in by_epoch])
        mean = samples.mean(axis=0)
        label, color, linestyle = METHOD_STYLES.get(method, (method, "#8C564B", "--"))
        ax.plot(epochs, mean, color=color, linestyle=linestyle, label=label,
                marker="o" if len(epochs) <= 2 else None, markersize=3)
        if show_std and len(samples) > 1:
            std = samples.std(axis=0, ddof=1)
            ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.13, linewidth=0)
    ax.margins(x=0.04)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=5, integer=True))


def save_figure(fig, root, name):
    for extension in ("png", "pdf"):
        fig.savefig(Path(root) / f"{name}.{extension}", dpi=300, bbox_inches="tight")
    if plt.get_backend().lower() != "agg":
        plt.show()
    plt.close(fig)


def plot_rank_trajectories(root, runs=None, show_std=False):
    """Аналог Figure 5: строки — сети, столбцы — группы 2, 3, 4."""
    runs = load_plot_runs(root) if runs is None else runs
    architectures = ("resnet18", "resnet50")
    with plt.rc_context(PAPER_STYLE):
        fig, axes = plt.subplots(2, 3, figsize=(9.0, 5.2), squeeze=False)
        for row_index, architecture in enumerate(architectures):
            selected = [r for r in runs if r["config"]["architecture"] == architecture]
            for column, layer in enumerate(("layer2", "layer3", "layer4")):
                ax = axes[row_index, column]
                if selected:
                    plot_metric(ax, selected, f"rank_{layer}", show_std)
                else:
                    ax.text(0.5, 0.5, "No recorded run", ha="center", va="center", transform=ax.transAxes)
                    ax.set_xticks([])
                    ax.set_yticks([])
                if column == 0:
                    ax.set_ylabel("Stable Rank")
                if row_index == 1:
                    ax.set_xlabel("Epochs")
                if row_index == 0:
                    ax.set_title(f"Group {column + 2}", fontsize=10, pad=7)
            axes[row_index, 0].annotate(
                f"{architecture.replace('resnet', 'ResNet-')} / CIFAR-10",
                xy=(-0.34, 0.5), xycoords="axes fraction", rotation=90,
                ha="center", va="center", fontsize=9,
            )
        # Одна легенда, как в Figure 5; при отсутствии первой строки — вторая.
        for ax in axes[:, 0]:
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(handles, labels, loc="upper right")
                break
        fig.subplots_adjust(left=0.12, right=0.99, bottom=0.10, top=0.94, hspace=0.30, wspace=0.25)
        save_figure(fig, root, "figure5-stable-rank")


def plot_training_curves(root, runs, show_std=False):
    # Дополнительная диагностика отдельно от композиции Figure 5.
    with plt.rc_context(PAPER_STYLE):
        for architecture in sorted({r["config"]["architecture"] for r in runs}):
            selected = [r for r in runs if r["config"]["architecture"] == architecture]
            fig, axes = plt.subplots(1, 3, figsize=(9.0, 2.8))
            eval_metric = "test_accuracy" if all("test_accuracy" in r["history"][0] for r in selected) else "val_accuracy"
            eval_label = "Test Accuracy (%)" if eval_metric == "test_accuracy" else "Validation Accuracy (%)"
            for ax, metric, label in zip(axes, ("train_loss", eval_metric, "train_accuracy"),
                                        ("Train Loss", eval_label, "Train Accuracy (%)")):
                plot_metric(ax, selected, metric, show_std)
                ax.set(xlabel="Epochs", ylabel=label)
            axes[0].legend(loc="best")
            fig.suptitle(f"{architecture.replace('resnet', 'ResNet-')} / CIFAR-10", fontsize=11)
            fig.tight_layout()
            save_figure(fig, root, f"{architecture}-training-curves")


def plot_results(root, show_std=False):
    root = Path(root)
    runs = load_plot_runs(root)
    plot_rank_trajectories(root, runs, show_std)
    plot_training_curves(root, runs, show_std)
    records = [r["summary"] for r in runs if r["summary"] is not None]
    aggregate = []
    for architecture, method in sorted({(r["architecture"], r["method"]) for r in records}):
        errors = [r["test_error"] for r in records
                  if r["architecture"] == architecture and r["method"] == method]
        aggregate.append({"architecture": architecture, "method": method, "runs": len(errors),
                          "mean_test_error": float(np.mean(errors)),
                          "std_test_error": float(np.std(errors, ddof=1)) if len(errors) > 1 else None})
    # Отдельная колонка — опубликованный ориентир, не измерение текущего запуска.
    reference_errors = {"zero": 5.13, "kaiming": 5.15, "xavier": 5.23}  # Table 2, CIFAR-10 / ResNet-18.
    for row in aggregate:
        reference = reference_errors.get(row["method"]) if row["architecture"] == "resnet18" else None
        row["paper_reference_error"] = reference
        row["difference_pp"] = row["mean_test_error"] - reference if reference is not None else None
    if aggregate:
        with (root / "aggregate.csv").open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(aggregate[0]))
            writer.writeheader()
            writer.writerows(aggregate)
        for row in aggregate:
            spread = f" ± {row['std_test_error']:.2f}" if row["std_test_error"] is not None else " (std: n/a)"
            print(f"{row['architecture']:8} {row['method']:8} test error: "
                  f"{row['mean_test_error']:.2f}{spread}% | n={row['runs']}")
    else:
        print("Обучение ещё не завершено: показана история, test-метрик пока нет.")
    return aggregate


In [ ]:
aggregate = plot_results(results_dir)
# Сохранённый или выполняющийся запуск:
# aggregate = plot_results(Path("runs/zero-init/ИМЯ_КАТАЛОГА"))
# Для продолжения: results_dir, summaries = run_experiments(config, resume=results_dir)


## Как сформулировать вывод после полного запуска

1. Сравните test error ZerO, Kaiming и Xavier при одинаковом фактически выбранном бюджете обучения (`runtime_plan.json`).
2. Проверьте кривые loss: обучается ли ZerO и насколько быстро относительно baseline.
3. Посмотрите на stable rank выбранных слоёв: отличаются ли начальные значения и траектории.
4. При нескольких seed сравните стандартное отклонение test error. Один запуск не проверяет воспроизводимость.
5. При необходимости добавьте `identity` и `zeros`, чтобы отдельно проверить роль Hadamard и последствие полного обнуления весов.

Результаты короткой проверки нельзя использовать для вывода о превосходстве одного метода. Обучение без BatchNorm здесь не проверяется. Для оценки после pruning / low-rank выполните отдельный анализ Figure 6.

## Figure 6: ранги обученных ядер, pruning и Tucker-2

Это отдельное измерение по `last.pt`, а не преобразование `history.json`. Повторное обучение не нужно, но для каждой точки compression требуется новый forward по test.

Из каталога `hw2`:

```bash
python figure6.py runs/zero-init/ИМЯ_КАТАЛОГА --device cuda --num-workers 4
```

Результат: `<каталог>/figure6/figure6-compression.png` и `.pdf`; исходные измерения — `measurements.json`. На CPU также работает (`--device cpu`), но медленнее. Только перерисовка без вычисления accuracy: `python figure6.py <каталог> --plot-only`.

- **Слева:** stable rank 17 свёрток ResNet-18, включая stem, без shortcut-проекций; развёртка `[Cout, Cin*kH*kW]`.
- **По центру:** отдельно в каждом Conv/Linear сохраняем 90–50% весов с наибольшим модулем. BN и bias сохраняются. Ось показывает долю оставленных позиций весов; исходные нули также могут занимать эти позиции.
- **Справа:** Tucker-2 по двум осям каналов, `r_out=r_in=r`, для `layer4.1.conv2` (512→512), ранги от 60 до 5. HOSVD-инициализация + 5 итераций HOOI; для измерения accuracy восстанавливается плотное приближённое ядро. Это не замер ускорения inference.

Все точки начинаются с одной исходной обученной модели, без накопления pruning, fine-tuning или обновления BN-статистик. Конкретный 512-канальный слой, параметры декомпозиции и часть деталей pruning в статье не раскрыты; наши соглашения записываются в JSON. Слой можно изменить через `--layer`.

В [Figure 6 статьи](https://arxiv.org/pdf/2110.12661v3#page=11) показаны три метода. В `h100_2h` по умолчанию только ZerO и Kaiming. Если нужны все три линии, перед обучением используйте `python run_zero.py --preset h100_2h --device cuda --methods zero kaiming xavier`; планировщик оценит время трёх запусков. Отсутствующая модель не заменяется нарисованной кривой.
